# 10. LLM Natural Language Explanation
**목적**: SHAP 기여도 수치를 OpenAI GPT-4o-mini로 자연어 설명으로 변환한다.

역할 구분:
- **SHAP**: 수학적으로 증명된 feature 기여도 (Explainable AI의 본체)
- **LLM**: SHAP 결과를 현장 엔지니어가 읽을 수 있는 언어로 번역하는 리포팅 도구

> ⚠️ LLM 출력은 SHAP 수치를 벗어나지 않도록 프롬프트를 엄격하게 제한함

In [ ]:
import sys
sys.path.append('..')

import os
import json
import pickle
import pandas as pd
import numpy as np
from pathlib import Path

from config import DATA_PROCESSED, OUT_TABLES

# OpenAI API 키 설정
# 방법 1: 환경변수 (권장)
# export OPENAI_API_KEY=sk-...
#
# 방법 2: 직접 입력 (보고서 제출 전 제거)
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')  # 여기에 직접 입력 가능

if not OPENAI_API_KEY:
    print('⚠ OPENAI_API_KEY가 없습니다.')
    print('  터미널에서: export OPENAI_API_KEY=sk-...')
    print('  또는 위 변수에 직접 입력')
else:
    print('✓ API 키 확인됨')

## 1. SHAP 결과 로드

In [ ]:
with open(DATA_PROCESSED / 'column_map.json', encoding='utf-8') as f:
    cmap = json.load(f)
FLAGS  = cmap['flags']
FC     = cmap['facility']
FID_COL = FC['facility_id']

df_risk = pd.read_parquet(DATA_PROCESSED / 'risk_scores.parquet')

# Case A: TreeExplainer SHAP
if FLAGS['has_label'] and (DATA_PROCESSED / 'best_model.pkl').exists():
    MODE = 'supervised'
    print('Case A: Supervised SHAP 사용')

# Case B: KernelExplainer SHAP
elif (DATA_PROCESSED / 'shap_kernel_results.pkl').exists():
    MODE = 'rule_based'
    with open(DATA_PROCESSED / 'shap_kernel_results.pkl', 'rb') as f:
        shap_data = pickle.load(f)
    print(f'Case B: KernelExplainer SHAP 사용')
    print(f'  설비 수: {len(shap_data["facility_ids"])}개')
    print(f'  feature: {shap_data["feature_cols"]}')

else:
    MODE = 'decomp_only'
    print('Score Decomposition만 사용 (SHAP 미실행)')

## 2. 설비 컨텍스트 구성 함수

In [ ]:
# feature 한국어 설명 매핑
FEATURE_KR = {
    'weather_hazard':        '기상위험도',
    'spatial_exposure':      '공간노출도(산림·지형)',
    'facility_exposure':     '설비노출도(밀집·노후)',
    'hist_prior':            '과거화재이력',
    'humidity_min_3d':       '최근3일 최저습도',
    'wind_max_1d':           '일최대풍속',
    'forest_ratio_500m':     '반경500m 산림비율',
    'slope_mean_500m':       '반경500m 평균경사도',
    'no_rain_days':          '연속무강수일수',
    'facility_density_1000m':'반경1km 설비밀도',
    'dry_watch_flag':        '건조주의보 조건',
    'combined_risk_flag':    '건조+강풍 복합위험',
    'elec_fire_count_1000m': '반경1km 전기화재이력',
    'fire_count_3000m':      '반경3km 산불이력',
}

GRADE_KR = {
    'Very High': '매우높음(긴급점검 필요)',
    'High':      '높음(우선점검 권고)',
    'Moderate':  '보통(모니터링 필요)',
    'Low':       '낮음(일반관리)',
}

def build_facility_context(facility_id, risk_row, shap_contributions):
    """설비 정보 + SHAP 기여도를 프롬프트용 컨텍스트로 변환."""
    top3 = sorted(shap_contributions.items(), key=lambda x: -abs(x[1]))[:3]
    
    contrib_text = '\n'.join([
        f"  - {FEATURE_KR.get(k, k)}: {'+' if v > 0 else ''}{v:.1f}점 영향"
        for k, v in top3
    ])
    
    context = {
        'facility_id': facility_id,
        'risk_score': round(risk_row['final_risk'], 1),
        'risk_grade': GRADE_KR.get(risk_row['risk_grade'], risk_row['risk_grade']),
        'weather': round(risk_row.get('weather_hazard', 0), 1),
        'spatial': round(risk_row.get('spatial_exposure', 0), 1),
        'facility': round(risk_row.get('facility_exposure', 0), 1),
        'prior': round(risk_row.get('hist_prior', 0), 1),
        'top3_contrib': contrib_text,
    }
    return context

print('컨텍스트 빌더 준비 완료')

## 3. OpenAI 자연어 설명 생성

In [ ]:
def generate_risk_explanation(ctx: dict, api_key: str, model: str = 'gpt-4o-mini') -> str:
    """
    SHAP 기여도를 받아 한국어 자연어 위험 설명을 생성한다.
    SHAP 수치를 벗어나는 할루시네이션을 방지하기 위해 엄격한 프롬프트 사용.
    """
    from openai import OpenAI
    client = OpenAI(api_key=api_key)
    
    system_prompt = """당신은 한국전력공사(KEPCO) 전력설비 화재위험 분석 시스템의 보고서 생성 모듈입니다.
    반드시 아래 규칙을 따르세요:
    1. 제공된 수치 외의 정보를 추가하거나 추측하지 마세요.
    2. 3~4문장으로 간결하게 작성하세요.
    3. 첫 문장: 위험등급과 주요 원인 요약
    4. 두 번째 문장: 가장 큰 기여 요인 구체적 설명
    5. 세 번째 문장: 권고 조치"""

    user_prompt = f"""다음 전력설비의 화재위험 분석 결과를 바탕으로 점검 담당자가 이해할 수 있는 설명을 작성하세요.

설비 ID: {ctx['facility_id']}
최종 위험도: {ctx['risk_score']}/100점
위험등급: {ctx['risk_grade']}

구성요소별 위험도:
  - 기상위험도: {ctx['weather']}점
  - 공간노출도(산림·지형): {ctx['spatial']}점
  - 설비노출도: {ctx['facility']}점
  - 과거화재이력: {ctx['prior']}점

SHAP 분석 - 주요 기여 요인 Top 3:
{ctx['top3_contrib']}"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=0.2,   # 일관성 ↑
        max_tokens=250,
    )
    return response.choices[0].message.content.strip()


# API 키 없을 때 Mock 출력
def mock_explanation(ctx: dict) -> str:
    return (
        f"설비 {ctx['facility_id']}는 {ctx['risk_grade']}으로 분류되었습니다. "
        f"기상위험도({ctx['weather']}점)와 공간노출도({ctx['spatial']}점)가 주요 위험 요인이며, "
        f"특히 {ctx['top3_contrib'].split(chr(10))[0].strip()}.  "
        f"즉각적인 현장 점검 및 절연 저항 측정이 권고됩니다."
    )

print('설명 생성 함수 준비 완료')

## 4. Top-10 고위험 설비 설명 생성

In [ ]:
# 최신 날짜 기준 Top-10 설비
latest = df_risk.sort_values('date').groupby(FID_COL).last().reset_index()
top10 = latest.nlargest(10, 'final_risk')

results = []
for _, row in top10.iterrows():
    fid = row[FID_COL]

    # SHAP 기여도 구성
    if MODE == 'rule_based' and fid in shap_data['facility_ids']:
        idx = list(shap_data['facility_ids']).index(fid)
        sv = shap_data['shap_values'][idx]
        contributions = dict(zip(shap_data['feature_cols'], sv))
    else:
        # Decomposition fallback
        contributions = {
            'weather_hazard':    row.get('weather_hazard', 0)    * 0.35,
            'spatial_exposure':  row.get('spatial_exposure', 0)  * 0.30,
            'facility_exposure': row.get('facility_exposure', 0) * 0.25,
            'hist_prior':        row.get('hist_prior', 0)        * 0.10,
        }

    ctx = build_facility_context(fid, row, contributions)

    # 설명 생성
    if OPENAI_API_KEY:
        try:
            explanation = generate_risk_explanation(ctx, OPENAI_API_KEY)
        except Exception as e:
            print(f'  ⚠ API 오류 ({fid}): {e}')
            explanation = mock_explanation(ctx)
    else:
        explanation = mock_explanation(ctx)

    results.append({
        FID_COL: fid,
        'risk_score': ctx['risk_score'],
        'risk_grade': row['risk_grade'],
        'explanation': explanation,
        **{f'shap_{k}': v for k, v in contributions.items()},
    })
    print(f'✓ {fid} ({ctx["risk_score"]}점): 설명 생성됨')

df_explained = pd.DataFrame(results)
df_explained.to_csv(OUT_TABLES / 'top10_llm_explanation.csv', index=False, encoding='utf-8-sig')

print('\n=== 생성된 설명 예시 (Top-1) ===')
print(f"설비: {df_explained.iloc[0][FID_COL]}")
print(f"위험도: {df_explained.iloc[0]['risk_score']}점")
print(f"\n{df_explained.iloc[0]['explanation']}")

## 5. 배치 처리 함수 (Streamlit 앱용)

In [ ]:
def get_explanation_for_facility(facility_id: str, api_key: str = OPENAI_API_KEY) -> dict:
    """
    Streamlit 앱에서 호출하는 단일 설비 설명 생성 함수.
    캐시 지원: 동일 설비는 재계산하지 않음.
    """
    # 캐시 확인
    cache_path = OUT_TABLES / 'explanation_cache.json'
    cache = {}
    if cache_path.exists():
        with open(cache_path, encoding='utf-8') as f:
            cache = json.load(f)

    if facility_id in cache:
        return cache[facility_id]

    # 설비 정보 조회
    latest = df_risk.sort_values('date').groupby(FID_COL).last().reset_index()
    row = latest[latest[FID_COL] == facility_id]
    if len(row) == 0:
        return {'explanation': '설비 정보를 찾을 수 없습니다.'}

    row = row.iloc[0]
    contributions = {
        'weather_hazard':    row.get('weather_hazard', 0)    * 0.35,
        'spatial_exposure':  row.get('spatial_exposure', 0)  * 0.30,
        'facility_exposure': row.get('facility_exposure', 0) * 0.25,
        'hist_prior':        row.get('hist_prior', 0)        * 0.10,
    }
    ctx = build_facility_context(facility_id, row, contributions)

    if api_key:
        explanation = generate_risk_explanation(ctx, api_key)
    else:
        explanation = mock_explanation(ctx)

    result = {'explanation': explanation, **ctx}

    # 캐시 저장
    cache[facility_id] = result
    with open(cache_path, 'w', encoding='utf-8') as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

    return result

print('Streamlit 연동 함수 준비 완료')
print('\n사용법:')
print('  from notebooks.10_llm_explanation import get_explanation_for_facility')
print('  result = get_explanation_for_facility("F-1023")')
print('  print(result["explanation"])')